In [1]:
from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn import metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, balanced_accuracy_score
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier


from keras import models, Input
from keras import optimizers as opt
from keras import backend as K
from keras.layers import Dense
from keras_tuner.tuners import RandomSearch
from tensorflow.keras.utils import to_categorical
import os
import joblib
import numpy as np
import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multitest import multipletests


In [2]:
from dataset import load_dataset, load_labels, split_data, format_labels
from features import (time_series_features, fractal_features, entropy_features,
                      hjorth_features, freq_band_features, epocx_bandpower_hjorth_features,
                      epocx_relative_power_hjorth_features, epocx_relative_power_hjorth_features_plus)
import variables as v

# Variables

In [3]:
data_type = "ica_filtered"
test_type = "Arithmetic"

# Load Dataset

In [4]:
dataset_ = load_dataset(data_type=data_type, test_type=test_type)
dataset = split_data(dataset_, v.SFREQ)

Selected channel numbers: [3, 4, 5, 8, 10, 16, 19, 25, 27, 30, 31, 32]
Selected SAM40 channel names: ['Fp1', 'F7', 'F3', 'FC5', 'T7', 'O1', 'O2', 'T8', 'FC6', 'F4', 'F8', 'Fp2']
Mapped EPOC X channel names: ['POW.AF3', 'POW.F7', 'POW.F3', 'POW.FC5', 'POW.T7', 'POW.O1', 'POW.O2', 'POW.T8', 'POW.FC6', 'POW.F4', 'POW.F8', 'POW.AF4']
Selected Python indices: [2, 3, 4, 7, 9, 15, 18, 24, 26, 29, 30, 31]
Expected dataset shape: (120, 12, 3200)
First loaded file: Arithmetic_sub_10_trial1.mat
Original data shape: (32, 3200)
Original channel count: 32
Original sample count: 3200
Selected data shape: (12, 3200)
Selected channels:
  original number 3, Python index 2, SAM40 name Fp1, EPOC X name POW.AF3
  original number 4, Python index 3, SAM40 name F7, EPOC X name POW.F7
  original number 5, Python index 4, SAM40 name F3, EPOC X name POW.F3
  original number 8, Python index 7, SAM40 name FC5, EPOC X name POW.FC5
  original number 10, Python index 9, SAM40 name T7, EPOC X name POW.T7
  original nu

In [9]:
# label_ = load_labels()
# label = format_labels(label_, test_type=test_type, epochs=dataset.shape[1])

stress         1281
mild_stress     819
high_stress     420
Name: count, dtype: int64
label shape: (2520,)
stress         1281
mild_stress     819
high_stress     420
Name: count, dtype: int64
['stress' 'mild_stress' 'high_stress']


# Compute Features

In [6]:
# features = time_series_features(dataset)
# freq_bands = np.array([1, 4, 8, 12, 30, 50])
# features = freq_band_features(dataset, freq_bands)
# features_hjorth = hjorth_features(dataset)
# features = entropy_features(dataset)
# features = fractal_features(dataset)
# features = epocx_bandwise_hjorth_entropy_features(dataset)
# print("features shape:", features.shape)
# print("number of samples:", features.shape[0])
# print("number of features:", features.shape[1])

Input data shape: (120, 25, 12, 128)
Output features shape: (3000, 540)
Features per channel: 45
NaN count: 0
Inf count: 0
features shape: (3000, 540)
number of samples: 3000
number of features: 540


## Map Feature names

In [5]:
# ========= 1. Generate feature names =========
def epocx_bandpower_hjorth_feature_names(channel_names):
    bands = [
        "theta",
        "alpha",
        "betaL",
        "betaH",
        "gamma"
    ]

    hjorth_features = [
        "hjorth_mobility",
        "hjorth_complexity"
    ]

    ratio_features = [
        "beta_alpha",
        "beta_alpha_theta",
        "theta_beta",
        "theta_alpha_beta",
        "inverse_alpha"
    ]

    names = []

    for ch in channel_names:
        for band in bands:
            for feat in hjorth_features:
                names.append(f"{ch}.{band}_power.{feat}")

        for ratio in ratio_features:
            names.append(f"{ch}.{ratio}")

    return names


def epocx_relative_power_hjorth_feature_names(channel_names):
    bands = [
        "theta",
        "alpha",
        "betaL",
        "betaH",
        "gamma"
    ]

    hjorth_features = [
        "hjorth_mobility",
        "hjorth_complexity"
    ]

    ratio_features = [
        "beta_alpha",
        "beta_alpha_theta",
        "theta_beta",
        "theta_alpha_beta",
        "inverse_alpha"
    ]

    names = []

    for ch in channel_names:
        for band in bands:
            for feat in hjorth_features:
                names.append(f"{ch}.{band}_relative_power.{feat}")

        for ratio in ratio_features:
            names.append(f"{ch}.relative_power.{ratio}")

    return names

def epocx_relative_power_hjorth_feature_plus_names(channel_names):
    bands = [
        "theta",
        "alpha",
        "betaL",
        "betaH",
        "gamma"
    ]

    hjorth_features = [
        "hjorth_mobility",
        "hjorth_complexity"
    ]

    ratio_features = [
        "beta_alpha",
        "beta_alpha_theta",
        "theta_beta",
        "theta_alpha_beta",
        "inverse_alpha"
    ]

    names = []

    for ch in channel_names:
        for band in bands:
            for feat in hjorth_features:
                names.append(f"{ch}.{band}_relative_power.{feat}")

        for ratio in ratio_features:
            for feat in hjorth_features:
                names.append(f"{ch}.{ratio}_relative_power_ratio.{feat}")

    return names
# ========= 2. Set channel names =========

channel_names = [
    "POW.AF3", "POW.F7", "POW.F3", "POW.FC5",
    "POW.T7", "POW.O1", "POW.O2", "POW.T8",
    "POW.FC6", "POW.F4", "POW.F8", "POW.AF4"
]

power_window = 5
step = 1

features = epocx_relative_power_hjorth_features(
    dataset,
    power_window=power_window,
    step=step
)

feature_names = epocx_relative_power_hjorth_feature_names(channel_names)

print("features shape:", features.shape)
print("number of feature names:", len(feature_names))

if features.shape[1] != len(feature_names):
    raise ValueError(
        f"features has {features.shape[1]} columns, "
        f"but feature_names has {len(feature_names)} names."
    )

n_windows = (dataset.shape[1] - power_window) // step + 1

label_ = load_labels()
label = format_labels(
    label_,
    test_type=test_type,
    epochs=n_windows
)

print("label shape:", label.shape)

if features.shape[0] != len(label):
    raise ValueError(
        f"features has {features.shape[0]} rows, "
        f"but label has {len(label)} rows."
    )
# ========= 3. Print feature examples =========

df_features = pd.DataFrame(features, columns=feature_names)
df_features["label"] = label

print("Feature DataFrame shape:", df_features.shape)

print("\nFirst 20 feature names:")
print(feature_names[:20])

print("\nLast 20 feature names:")
print(feature_names[-20:])

# ========= 4. Save features =========
# save_dir = "saved_features/epocx_relative_power_hjorth_features_plus"
# os.makedirs(save_dir, exist_ok=True)
#
# save_features = os.path.join(
#     save_dir,
#     "epocx_relative_power_hjorth_features.joblib"
# )
#
# save_feature_names = os.path.join(
#     save_dir,
#     "epocx_relative_power_hjorth_feature_names.joblib"
# )
#
# save_labels = os.path.join(
#     save_dir,
#     "epocx_relative_power_hjorth_labels.joblib"
# )
#
# save_features_csv = os.path.join(
#     save_dir,
#     "epocx_relative_power_hjorth_features_with_label.csv"
# )
#
# joblib.dump(features, save_features)
# joblib.dump(feature_names, save_feature_names)
# joblib.dump(label, save_labels)
#
# df_features.to_csv(save_features_csv, index=False)
#
# print(f"Saved features to {save_features}")
# print(f"Saved feature names to {save_feature_names}")
# print(f"Saved labels to {save_labels}")
# print(f"Saved features CSV to {save_features_csv}")

Absolute band power sequence shape: (120, 25, 12, 5)
Relative band power sequence shape: (120, 25, 12, 5)
Meaning: n_trials, n_secs, n_channels, n_bands
Input EEG data shape: (120, 25, 12, 128)
Output relative-power Hjorth features shape: (2520, 180)
Features per channel: 15
Power window: 5
Step: 1
Number of windows per trial: 21
NaN count: 0
Inf count: 0
features shape: (2520, 180)
number of feature names: 180
label shape: (2520,)
Feature DataFrame shape: (2520, 181)

First 20 feature names:
['POW.AF3.theta_relative_power.hjorth_mobility', 'POW.AF3.theta_relative_power.hjorth_complexity', 'POW.AF3.alpha_relative_power.hjorth_mobility', 'POW.AF3.alpha_relative_power.hjorth_complexity', 'POW.AF3.betaL_relative_power.hjorth_mobility', 'POW.AF3.betaL_relative_power.hjorth_complexity', 'POW.AF3.betaH_relative_power.hjorth_mobility', 'POW.AF3.betaH_relative_power.hjorth_complexity', 'POW.AF3.gamma_relative_power.hjorth_mobility', 'POW.AF3.gamma_relative_power.hjorth_complexity', 'POW.AF3.re

## Feature Selection

In [8]:
# ========= 3. Check label alignment =========

print("label length:", len(label))
print("label distribution:")
print(pd.Series(label).value_counts(dropna=False))

if features.shape[0] != len(label):
    raise ValueError(
        f"Sample mismatch: features has {features.shape[0]} rows, "
        f"but label has {len(label)} labels."
    )

# ========= 4. Build feature DataFrame =========

df_features = pd.DataFrame(features, columns=feature_names)
df_features["label"] = label

print("Feature DataFrame shape:", df_features.shape)
print(df_features.head())

# ========= 5. Clean data =========

LABEL_COL = "label"
feature_cols = feature_names

data_for_selection = df_features[[LABEL_COL] + feature_cols].copy()

data_for_selection = data_for_selection.dropna(subset=[LABEL_COL])

for col in feature_cols:
    data_for_selection[col] = pd.to_numeric(
        data_for_selection[col],
        errors="coerce"
    )

data_for_selection = data_for_selection.replace([np.inf, -np.inf], np.nan)
data_for_selection = data_for_selection.dropna(subset=feature_cols)

print("Data used for feature selection:", data_for_selection.shape)
print("Label distribution after cleaning:")
print(data_for_selection[LABEL_COL].value_counts(dropna=False))

# ========= 6. Run one-way ANOVA feature by feature =========

results = []

y = data_for_selection[LABEL_COL]
classes = sorted(y.unique())

if len(classes) < 2:
    raise ValueError("At least two classes are required to run ANOVA.")

for feat in feature_cols:
    groups = [
        data_for_selection.loc[y == cls, feat].values
        for cls in classes
    ]

    groups = [g for g in groups if len(g) > 1]

    if len(groups) < 2:
        continue

    F, p = f_oneway(*groups)

    x = data_for_selection[feat].values
    grand_mean = np.mean(x)

    ss_between = 0.0
    for cls in classes:
        g = data_for_selection.loc[y == cls, feat].values
        if len(g) == 0:
            continue
        ss_between += len(g) * (np.mean(g) - grand_mean) ** 2

    ss_total = np.sum((x - grand_mean) ** 2)
    eta2 = ss_between / ss_total if ss_total > 0 else np.nan

    results.append([feat, F, p, eta2])

res = pd.DataFrame(
    results,
    columns=["feature", "F", "p", "eta2"]
)

if len(res) == 0:
    raise ValueError("No valid features were available for ANOVA.")

# ========= 7. FDR-BH correction =========

rej, qvals, _, _ = multipletests(
    res["p"].values,
    alpha=0.05,
    method="fdr_bh"
)

res["q"] = qvals
res["significant_fdr"] = rej

# ========= 8. Rank and select features =========

res = res.sort_values(
    ["q", "F"],
    ascending=[True, False]
).reset_index(drop=True)
significant_sum = int(res["significant_fdr"].sum())

selected = res[
    (res["q"] < 0.05) &
    (res["eta2"] >= 0.01)
    ].copy()

selected_feature_names = res.head(significant_sum)["feature"].tolist()  # selected["feature"].tolist()

print("Total features:", len(res))
print("FDR-significant features:", int(res["significant_fdr"].sum()))
print("Final selected features:", len(selected_feature_names))

print("\nTop 30 features:")
print(res.head(30).to_string(index=False))

print("\nSelected features:")
print(selected_feature_names)

# ========= 9. Build selected feature matrix =========

selected_features = df_features[selected_feature_names].to_numpy()
selected_label = df_features["label"].to_numpy()

print("Selected feature matrix shape:", selected_features.shape)
print("Selected label shape:", selected_label.shape)

label length: 3000
label distribution:
stress         1525
mild_stress     975
high_stress     500
Name: count, dtype: int64
Feature DataFrame shape: (3000, 541)
   POW.AF3.theta.power  POW.AF3.theta.hjorth_activity  \
0             7.512371                       6.481739   
1             3.993913                       3.355359   
2             5.005832                       7.084762   
3             7.076519                       4.663085   
4            17.973388                      10.346797   

   POW.AF3.theta.hjorth_mobility  POW.AF3.theta.hjorth_complexity  \
0                     309.179039                     13603.584579   
1                     162.667625                      7699.085250   
2                     111.823609                      3922.968255   
3                     274.444589                     12223.663593   
4                     580.138710                     21755.857879   

   POW.AF3.theta.app_entropy  POW.AF3.theta.sample_entropy  \
0                 

## Data

In [6]:
data = features

# Data split

In [7]:
# ========= 0. Settings =========
power_window = 5
step = 1

n_subjects = 40
n_trials_per_subject = 3
n_secs_per_trial = dataset.shape[1]

n_windows_per_trial = (n_secs_per_trial - power_window) // step + 1

print("n_secs_per_trial:", n_secs_per_trial)
print("power_window:", power_window)
print("step:", step)
print("n_windows_per_trial:", n_windows_per_trial)

# ========= 1. Encode labels =========
label_mapping = {
    "mild_stress": 0,
    "stress": 1,
    "high_stress": 2
}

target_names = ["mild_stress", "stress", "high_stress"]

if pd.api.types.is_numeric_dtype(pd.Series(label)):
    label_encoded = np.asarray(label)
else:
    label_encoded = pd.Series(label).map(label_mapping).to_numpy()

if pd.isna(label_encoded).sum() > 0:
    print("Original label values:")
    print(pd.Series(label).value_counts(dropna=False))
    raise ValueError("label_encoded contains NaN. Please check label_mapping.")

print("Data shape:", data.shape)
print("Label shape:", label_encoded.shape)
print("Label distribution:")
print(pd.Series(label_encoded).value_counts(dropna=False).sort_index())

# ========= 2. Build subject IDs =========
subject_ids = np.repeat(
    np.arange(n_subjects),
    n_trials_per_subject * n_windows_per_trial
)

print("Subject IDs shape:", subject_ids.shape)
print("Number of subjects:", len(np.unique(subject_ids)))

expected_samples = n_subjects * n_trials_per_subject * n_windows_per_trial

print("Expected samples:", expected_samples)
print("Actual data rows:", data.shape[0])
print("Actual label rows:", len(label_encoded))

if len(subject_ids) != data.shape[0]:
    raise ValueError(
        f"subject_ids length {len(subject_ids)} != data rows {data.shape[0]}. "
        "Check power_window, step, and feature generation."
    )

if len(label_encoded) != data.shape[0]:
    raise ValueError(
        f"label length {len(label_encoded)} != data rows {data.shape[0]}. "
        "Check format_labels epochs."
    )

# ========= 3. Subject-level train_val / test split =========
gss_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=1
)

train_val_idx, test_idx = next(
    gss_test.split(data, label_encoded, groups=subject_ids)
)

x_train_val = data[train_val_idx]
y_train_val = label_encoded[train_val_idx]
groups_train_val = subject_ids[train_val_idx]

x_test = data[test_idx]
y_test = label_encoded[test_idx]
groups_test = subject_ids[test_idx]

# ========= 4. Subject-level train / valid split =========
gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.05,
    random_state=1
)

train_idx, val_idx = next(
    gss_val.split(
        x_train_val,
        y_train_val,
        groups=groups_train_val
    )
)

x_train = x_train_val[train_idx]
y_train = y_train_val[train_idx]
groups_train = groups_train_val[train_idx]

x_val = x_train_val[val_idx]
y_val = y_train_val[val_idx]
groups_val = groups_train_val[val_idx]

# ========= 5. Check subject separation =========
train_subjects = np.unique(groups_train)
val_subjects = np.unique(groups_val)
test_subjects = np.unique(groups_test)

print("Train subjects:", train_subjects)
print("Valid subjects:", val_subjects)
print("Test subjects:", test_subjects)

print("Train/valid overlap:", np.intersect1d(train_subjects, val_subjects))
print("Train/test overlap:", np.intersect1d(train_subjects, test_subjects))
print("Valid/test overlap:", np.intersect1d(val_subjects, test_subjects))

print("x_train:", x_train.shape)
print("x_val:", x_val.shape)
print("x_test:", x_test.shape)

print("y_train distribution:")
print(pd.Series(y_train).value_counts(dropna=False).sort_index())

print("y_val distribution:")
print(pd.Series(y_val).value_counts(dropna=False).sort_index())

print("y_test distribution:")
print(pd.Series(y_test).value_counts(dropna=False).sort_index())

n_secs_per_trial: 25
power_window: 5
step: 1
n_windows_per_trial: 21
Data shape: (2520, 180)
Label shape: (2520,)
Label distribution:
0     819
1    1281
2     420
Name: count, dtype: int64
Subject IDs shape: (2520,)
Number of subjects: 40
Expected samples: 2520
Actual data rows: 2520
Actual label rows: 2520
Train subjects: [ 0  1  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 22 23 24 25 26
 27 28 29 30 32 33 35 36 37 39]
Valid subjects: [34 38]
Test subjects: [ 2  3 21 31]
Train/valid overlap: []
Train/test overlap: []
Valid/test overlap: []
x_train: (2142, 180)
x_val: (126, 180)
x_test: (252, 180)
y_train distribution:
0     693
1    1092
2     357
Name: count, dtype: int64
y_val distribution:
0    21
1    63
2    42
Name: count, dtype: int64
y_test distribution:
0    105
1    126
2     21
Name: count, dtype: int64


# k-NN Classifier

In [18]:
# ========= 6. KNN + PCA parameter search using validation macro F1 =========
best_val_f1 = -1
best_val_acc = -1
best_val_balanced_acc = -1
best_params = None
best_model = None
best_scaler = None
best_pca = None

scaler_candidates = {
    "standard": StandardScaler(),
    "minmax": MinMaxScaler()
}

pca_candidates = [
    0.90,
    0.95,
    0.98,
    20,
    30,
    50,
    80,
    100
]

for scaler_name, scaler in scaler_candidates.items():
    x_train_scaled = scaler.fit_transform(x_train)
    x_val_scaled = scaler.transform(x_val)

    for pca_n in pca_candidates:
        pca = PCA(
            n_components=pca_n,
            random_state=1
        )

        x_train_pca = pca.fit_transform(x_train_scaled)
        x_val_pca = pca.transform(x_val_scaled)

        print(
            f"Scaler={scaler_name}, PCA={pca_n}, "
            f"components={x_train_pca.shape[1]}, "
            f"explained_var={pca.explained_variance_ratio_.sum():.4f}"
        )

        for n_neighbors in range(1, 31):
            for p in [1, 2]:
                for weights in ["uniform", "distance"]:
                    knn = KNeighborsClassifier(
                        n_neighbors=n_neighbors,
                        p=p,
                        weights=weights
                    )

                    knn.fit(x_train_pca, y_train)
                    y_val_pred = knn.predict(x_val_pca)

                    val_acc = accuracy_score(y_val, y_val_pred)
                    val_macro_f1 = f1_score(
                        y_val,
                        y_val_pred,
                        average="macro",
                        zero_division=0
                    )
                    val_balanced_acc = balanced_accuracy_score(
                        y_val,
                        y_val_pred
                    )

                    # use macro F1 as the primary selection criteria
                    if val_macro_f1 > best_val_f1:
                        best_val_f1 = val_macro_f1
                        best_val_acc = val_acc
                        best_val_balanced_acc = val_balanced_acc

                        best_params = {
                            "selection_metric": "macro_f1",
                            "scaler": scaler_name,
                            "pca_n_components": pca_n,
                            "actual_pca_components": x_train_pca.shape[1],
                            "pca_explained_variance": float(
                                pca.explained_variance_ratio_.sum()
                            ),
                            "n_neighbors": n_neighbors,
                            "p": p,
                            "weights": weights,
                            "val_accuracy": float(val_acc),
                            "val_macro_f1": float(val_macro_f1),
                            "val_balanced_accuracy": float(val_balanced_acc)
                        }

                        best_model = knn
                        best_scaler = scaler
                        best_pca = pca


print("Best validation macro F1:", best_val_f1)
print("Best validation accuracy:", best_val_acc)
print("Best validation balanced accuracy:", best_val_balanced_acc)
print("Best parameters:", best_params)

# ========= 7. Refit final model on train + valid =========
x_train_valid = np.concatenate([x_train, x_val], axis=0)
y_train_valid = np.concatenate([y_train, y_val], axis=0)

if best_params["scaler"] == "standard":
    final_scaler = StandardScaler()
else:
    final_scaler = MinMaxScaler()

x_train_valid_scaled = final_scaler.fit_transform(x_train_valid)
x_test_scaled = final_scaler.transform(x_test)

final_pca = PCA(
    n_components=best_params["pca_n_components"],
    random_state=1
)

x_train_valid_pca = final_pca.fit_transform(x_train_valid_scaled)
x_test_pca = final_pca.transform(x_test_scaled)

final_knn = KNeighborsClassifier(
    n_neighbors=best_params["n_neighbors"],
    p=best_params["p"],
    weights=best_params["weights"]
)

final_knn.fit(x_train_valid_pca, y_train_valid)

y_test_pred = final_knn.predict(x_test_pca)

test_acc = accuracy_score(y_test, y_test_pred)
test_macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)
test_weighted_f1 = f1_score(
    y_test,
    y_test_pred,
    average="weighted",
    zero_division=0
)
test_balanced_acc = balanced_accuracy_score(y_test, y_test_pred)

print("Test accuracy:", test_acc)
print("Test macro F1:", test_macro_f1)
print("Test weighted F1:", test_weighted_f1)
print("Test balanced accuracy:", test_balanced_acc)

print("Classification report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=target_names,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

# ========= 8. Save model package =========
save_dir = "models/knn"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "knn_pca_relative_bandpower_hjorth_model.joblib")

if len(feature_names) != data.shape[1]:
    raise ValueError(
        f"feature_names length {len(feature_names)} does not match "
        f"data columns {data.shape[1]}"
    )

model_package = {
    "model": final_knn,
    "scaler": final_scaler,
    "pca": final_pca,
    "best_params": best_params,
    "label_mapping": label_mapping,
    "target_names": target_names,
    "feature_names": feature_names,
    "feature_shape": data.shape,
    "power_window": power_window,
    "step": step,
    "n_windows_per_trial": n_windows_per_trial,
    "pca_input_features": data.shape[1],
    "pca_output_features": x_train_valid_pca.shape[1],
    "pca_explained_variance": float(final_pca.explained_variance_ratio_.sum()),
    "test_accuracy": test_acc
}

joblib.dump(model_package, model_path)

print("Model saved to:", model_path)
print("Saved original feature number:", len(feature_names))
print("Saved PCA feature number:", x_train_valid_pca.shape[1])
print("First 10 original features:", feature_names[:10])


Scaler=standard, PCA=0.9, components=94, explained_var=0.9006
Scaler=standard, PCA=0.95, components=116, explained_var=0.9510
Scaler=standard, PCA=0.98, components=135, explained_var=0.9812
Scaler=standard, PCA=20, components=20, explained_var=0.4806
Scaler=standard, PCA=30, components=30, explained_var=0.5739
Scaler=standard, PCA=50, components=50, explained_var=0.7128
Scaler=standard, PCA=80, components=80, explained_var=0.8555
Scaler=standard, PCA=100, components=100, explained_var=0.9165
Scaler=minmax, PCA=0.9, components=67, explained_var=0.9037
Scaler=minmax, PCA=0.95, components=83, explained_var=0.9505
Scaler=minmax, PCA=0.98, components=109, explained_var=0.9804
Scaler=minmax, PCA=20, components=20, explained_var=0.5482
Scaler=minmax, PCA=30, components=30, explained_var=0.6604
Scaler=minmax, PCA=50, components=50, explained_var=0.8166
Scaler=minmax, PCA=80, components=80, explained_var=0.9445
Scaler=minmax, PCA=100, components=100, explained_var=0.9728
Best validation macro F

# SVM Classifier

In [14]:
# ========= 6. KNN + PCA parameter search using validation macro F1 =========
best_val_f1 = -1
best_val_acc = -1
best_val_balanced_acc = -1
best_params = None
best_model = None
best_scaler = None
best_pca = None

scaler_candidates = {
    "standard": StandardScaler(),
    "minmax": MinMaxScaler()
}

pca_candidates = [
    None,
    20,
    30,
    50,
    80,
    100,
    0.90,
    0.95
]

C_candidates = [0.1, 1, 10]
kernel_candidates = ["linear", "rbf"]
gamma_candidates = ["scale", "auto"]

for scaler_name, scaler in scaler_candidates.items():
    x_train_scaled = scaler.fit_transform(x_train)
    x_val_scaled = scaler.transform(x_val)

    for pca_n in pca_candidates:
        if pca_n is None:
            x_train_model = x_train_scaled
            x_val_model = x_val_scaled
            pca = None
            actual_components = x_train_model.shape[1]
            explained_var = None
        else:
            pca = PCA(n_components=pca_n, random_state=1)
            x_train_model = pca.fit_transform(x_train_scaled)
            x_val_model = pca.transform(x_val_scaled)
            actual_components = x_train_model.shape[1]
            explained_var = float(pca.explained_variance_ratio_.sum())

        print(
            f"Scaler={scaler_name}, PCA={pca_n}, "
            f"components={actual_components}, explained_var={explained_var}"
        )

        for kernel in kernel_candidates:
            for C in C_candidates:
                for gamma in gamma_candidates:
                    if kernel == "linear" and gamma != "scale":
                        continue

                    svm = SVC(
                        kernel=kernel,
                        C=C,
                        gamma=gamma,
                        class_weight="balanced",
                        probability=False,
                        random_state=1
                    )

                    svm.fit(x_train_model, y_train)
                    y_val_pred = svm.predict(x_val_model)

                    val_acc = accuracy_score(y_val, y_val_pred)
                    val_macro_f1 = f1_score(
                        y_val,
                        y_val_pred,
                        average="macro",
                        zero_division=0
                    )
                    val_balanced_acc = balanced_accuracy_score(
                        y_val,
                        y_val_pred
                    )

                    if val_macro_f1 > best_val_f1:
                        best_val_f1 = val_macro_f1
                        best_val_acc = val_acc
                        best_val_balanced_acc = val_balanced_acc

                        best_params = {
                            "model_type": "SVM",
                            "selection_metric": "macro_f1",
                            "scaler": scaler_name,
                            "pca_n_components": pca_n,
                            "actual_pca_components": actual_components,
                            "pca_explained_variance": explained_var,
                            "kernel": kernel,
                            "C": C,
                            "gamma": gamma,
                            "class_weight": "balanced",
                            "val_accuracy": float(val_acc),
                            "val_macro_f1": float(val_macro_f1),
                            "val_balanced_accuracy": float(val_balanced_acc)
                        }

                        best_model = svm
                        best_scaler = scaler
                        best_pca = pca

print("Best validation macro F1:", best_val_f1)
print("Best validation accuracy:", best_val_acc)
print("Best validation balanced accuracy:", best_val_balanced_acc)
print("Best parameters:", best_params)

# ========= 7. Refit final model on train + valid =========
x_train_valid = np.concatenate([x_train, x_val], axis=0)
y_train_valid = np.concatenate([y_train, y_val], axis=0)

if best_params["scaler"] == "standard":
    final_scaler = StandardScaler()
else:
    final_scaler = MinMaxScaler()

x_train_valid_scaled = final_scaler.fit_transform(x_train_valid)
x_test_scaled = final_scaler.transform(x_test)

if best_params["pca_n_components"] is None:
    final_pca = None
    x_train_valid_model = x_train_valid_scaled
    x_test_model = x_test_scaled
else:
    final_pca = PCA(
        n_components=best_params["pca_n_components"],
        random_state=1
    )
    x_train_valid_model = final_pca.fit_transform(x_train_valid_scaled)
    x_test_model = final_pca.transform(x_test_scaled)

final_svm = SVC(
    kernel=best_params["kernel"],
    C=best_params["C"],
    gamma=best_params["gamma"],
    class_weight="balanced",
    probability=False,
    random_state=1
)

final_svm.fit(x_train_valid_model, y_train_valid)

y_test_pred = final_svm.predict(x_test_model)

test_acc = accuracy_score(y_test, y_test_pred)
test_macro_f1 = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
test_weighted_f1 = f1_score(y_test, y_test_pred, average="weighted", zero_division=0)
test_balanced_acc = balanced_accuracy_score(y_test, y_test_pred)

print("Test accuracy:", test_acc)
print("Test macro F1:", test_macro_f1)
print("Test weighted F1:", test_weighted_f1)
print("Test balanced accuracy:", test_balanced_acc)

print("Classification report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=target_names,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_test_pred))


Scaler=standard, PCA=None, components=180, explained_var=None
Scaler=standard, PCA=20, components=20, explained_var=0.4806449788198858
Scaler=standard, PCA=30, components=30, explained_var=0.5738920317305103
Scaler=standard, PCA=50, components=50, explained_var=0.7128481098456553
Scaler=standard, PCA=80, components=80, explained_var=0.8555076038116108
Scaler=standard, PCA=100, components=100, explained_var=0.9165243819199982
Scaler=standard, PCA=0.9, components=94, explained_var=0.900583177139951
Scaler=standard, PCA=0.95, components=116, explained_var=0.9509980394511195
Scaler=minmax, PCA=None, components=180, explained_var=None
Scaler=minmax, PCA=20, components=20, explained_var=0.548193511119898
Scaler=minmax, PCA=30, components=30, explained_var=0.6603998846199168
Scaler=minmax, PCA=50, components=50, explained_var=0.81663401570763
Scaler=minmax, PCA=80, components=80, explained_var=0.9445398879947886
Scaler=minmax, PCA=100, components=100, explained_var=0.97277792805703
Scaler=min

In [15]:
# ========= Save SVM model with PCA =========
save_dir = "models/svm"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(
    save_dir,
    "svm_relative_power_hjorth_model.joblib"
)

if len(feature_names) != data.shape[1]:
    raise ValueError(
        f"feature_names length {len(feature_names)} does not match "
        f"data columns {data.shape[1]}"
    )

model_package = {
    "model": final_svm,
    "scaler": final_scaler,
    "pca": final_pca,

    "model_type": "SVM_RBF_PCA",
    "params": {
        "kernel": best_params["kernel"],
        "C": best_params["C"],
        "gamma": best_params["gamma"],
        "class_weight": "balanced"
    },

    "label_mapping": label_mapping,
    "target_names": target_names,
    "feature_names": feature_names,
    "feature_shape": data.shape,

    "feature_type": "relative_power_hjorth_ratio_hjorth",
    "power_window": power_window,
    "step": step,
    "n_windows_per_trial": n_windows_per_trial,

    "pca_input_features": data.shape[1],
    "pca_output_features": final_pca.n_components_,
    "pca_explained_variance": float(final_pca.explained_variance_ratio_.sum()),

    "test_accuracy": float(test_acc),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1),
    "test_balanced_accuracy": float(test_balanced_acc)
}

joblib.dump(model_package, model_path)

print("Model saved to:", model_path)
print("Saved original feature number:", len(feature_names))
print("Saved PCA feature number:", final_pca.n_components_)
print("Saved PCA explained variance:", float(final_pca.explained_variance_ratio_.sum()))
print("First 10 feature names:")
print(feature_names[:10])

Model saved to: models/svm\svm_relative_power_hjorth_model.joblib
Saved original feature number: 180
Saved PCA feature number: 30
Saved PCA explained variance: 0.5730503336591258
First 10 feature names:
['POW.AF3.theta_relative_power.hjorth_mobility', 'POW.AF3.theta_relative_power.hjorth_complexity', 'POW.AF3.alpha_relative_power.hjorth_mobility', 'POW.AF3.alpha_relative_power.hjorth_complexity', 'POW.AF3.betaL_relative_power.hjorth_mobility', 'POW.AF3.betaL_relative_power.hjorth_complexity', 'POW.AF3.betaH_relative_power.hjorth_mobility', 'POW.AF3.betaH_relative_power.hjorth_complexity', 'POW.AF3.gamma_relative_power.hjorth_mobility', 'POW.AF3.gamma_relative_power.hjorth_complexity']


# Load SVM

In [16]:
model_package = joblib.load(
    "models/svm/svm_relative_power_hjorth_model.joblib"
)

model = model_package["model"]
scaler = model_package["scaler"]
pca = model_package["pca"]
feature_names = model_package["feature_names"]
label_mapping = model_package["label_mapping"]

print(pca)
print(scaler)
print(model_package["model_type"])
print(model_package["params"])
print(len(feature_names))

PCA(n_components=30, random_state=1)
StandardScaler()
SVM_RBF_PCA
{'kernel': 'rbf', 'C': 1, 'gamma': 'scale', 'class_weight': 'balanced'}
180


# SVM (no pca, set single combination of parameters)

In [17]:
# ========= 6. Single RBF SVM baseline, no PCA =========
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)
x_test_scaled = scaler.transform(x_test)
C = 100
gamma = "scale"
class_weights = "balanced"
kernel = "rbf"
svm = SVC(
    kernel=kernel,
    C=C,
    gamma=gamma,
    class_weight=class_weights,
    random_state=1
)

svm.fit(x_train_scaled, y_train)


# ========= 7. Validation evaluation =========

y_val_pred = svm.predict(x_val_scaled)

val_acc = accuracy_score(y_val, y_val_pred)
val_macro_f1 = f1_score(
    y_val,
    y_val_pred,
    average="macro",
    zero_division=0
)
val_weighted_f1 = f1_score(
    y_val,
    y_val_pred,
    average="weighted",
    zero_division=0
)
val_balanced_acc = balanced_accuracy_score(y_val, y_val_pred)

print("Validation accuracy:", val_acc)
print("Validation macro F1:", val_macro_f1)
print("Validation weighted F1:", val_weighted_f1)
print("Validation balanced accuracy:", val_balanced_acc)

print("Validation classification report:")
print(classification_report(
    y_val,
    y_val_pred,
    target_names=target_names,
    zero_division=0
))

print("Validation confusion matrix:")
print(confusion_matrix(y_val, y_val_pred))


# ========= 8. Test evaluation =========

y_test_pred = svm.predict(x_test_scaled)

test_acc = accuracy_score(y_test, y_test_pred)
test_macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)
test_weighted_f1 = f1_score(
    y_test,
    y_test_pred,
    average="weighted",
    zero_division=0
)
test_balanced_acc = balanced_accuracy_score(y_test, y_test_pred)

print("Test accuracy:", test_acc)
print("Test macro F1:", test_macro_f1)
print("Test weighted F1:", test_weighted_f1)
print("Test balanced accuracy:", test_balanced_acc)

print("Test classification report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=target_names,
    zero_division=0
))

print("Test confusion matrix:")
print(confusion_matrix(y_test, y_test_pred))


# ========= 10. Save model =========

save_dir = "models/svm"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, f"svm_{kernel}_stress_model.joblib")

model_package = {
    "model": svm,
    "scaler": scaler,
    "pca": None,
    "model_type": f"svm_{kernel}",
    "params": {
        "kernel": kernel,
        "C": C,
        "gamma": gamma,
        "class_weight": class_weights,
    },
    "label_mapping": label_mapping,
    "target_names": target_names,
    "feature_names": feature_names,
    "feature_shape": data.shape,
    "power_window": power_window,
    "step": step,
    "n_windows_per_trial": n_windows_per_trial,
    "val_accuracy": float(val_acc),
    "val_macro_f1": float(val_macro_f1),
    "val_weighted_f1": float(val_weighted_f1),
    "val_balanced_accuracy": float(val_balanced_acc),
    "test_accuracy": float(test_acc),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1),
    "test_balanced_accuracy": float(test_balanced_acc)
}

if len(feature_names) != data.shape[1]:
    raise ValueError(
        f"feature_names length {len(feature_names)} does not match "
        f"data columns {data.shape[1]}"
    )

joblib.dump(model_package, model_path)

print("Model saved to:", model_path)

Validation accuracy: 0.3492063492063492
Validation macro F1: 0.2538690476190476
Validation weighted F1: 0.32872023809523804
Validation balanced accuracy: 0.2724867724867725
Validation classification report:
              precision    recall  f1-score   support

 mild_stress       0.07      0.14      0.09        21
      stress       0.49      0.60      0.54        63
 high_stress       0.50      0.07      0.12        42

    accuracy                           0.35       126
   macro avg       0.35      0.27      0.25       126
weighted avg       0.43      0.35      0.33       126

Validation confusion matrix:
[[ 3 18  0]
 [22 38  3]
 [18 21  3]]
Test accuracy: 0.44047619047619047
Test macro F1: 0.3329222457129434
Test weighted F1: 0.4193093727977449
Test balanced accuracy: 0.34603174603174597
Test classification report:
              precision    recall  f1-score   support

 mild_stress       0.36      0.23      0.28       105
      stress       0.52      0.67      0.58       126
 high

# RF

In [8]:
from sklearn.ensemble import RandomForestClassifier
best_val_f1 = -1
best_params = None
best_model = None

n_estimators_candidates = [200, 500, 800]
max_depth_candidates = [None, 5, 10, 20]
min_samples_leaf_candidates = [1, 3, 5, 10]
max_features_candidates = ["sqrt", "log2"]

for n_estimators in n_estimators_candidates:
    for max_depth in max_depth_candidates:

        for min_samples_leaf in min_samples_leaf_candidates:
            for max_features in max_features_candidates:

                rf = RandomForestClassifier(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_leaf=min_samples_leaf,
                    max_features=max_features,
                    class_weight="balanced",
                    random_state=1,
                    n_jobs=-1
                )

                rf.fit(x_train, y_train)
                y_val_pred = rf.predict(x_val)

                val_acc = accuracy_score(y_val, y_val_pred)
                val_macro_f1 = f1_score(
                    y_val,
                    y_val_pred,
                    average="macro",
                    zero_division=0
                )
                val_balanced_acc = balanced_accuracy_score(
                    y_val,
                    y_val_pred
                )

                if val_macro_f1 > best_val_f1:
                    best_val_f1 = val_macro_f1
                    best_params = {
                        "model_type": "RandomForest",
                        "selection_metric": "macro_f1",
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_leaf": min_samples_leaf,
                        "max_features": max_features,
                        "class_weight": "balanced",
                        "val_accuracy": float(val_acc),
                        "val_macro_f1": float(val_macro_f1),
                        "val_balanced_accuracy": float(val_balanced_acc)
                    }
                    best_model = rf

print("Best validation macro F1:", best_val_f1)
print("Best parameters:", best_params)

x_train_valid = np.concatenate([x_train, x_val], axis=0)
y_train_valid = np.concatenate([y_train, y_val], axis=0)

final_rf = RandomForestClassifier(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    min_samples_leaf=best_params["min_samples_leaf"],
    max_features=best_params["max_features"],
    class_weight="balanced",
    random_state=1,
    n_jobs=-1
)

final_rf.fit(x_train_valid, y_train_valid)

y_test_pred = final_rf.predict(x_test)

test_acc = accuracy_score(y_test, y_test_pred)
test_macro_f1 = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
test_weighted_f1 = f1_score(y_test, y_test_pred, average="weighted", zero_division=0)
test_balanced_acc = balanced_accuracy_score(y_test, y_test_pred)

print("Test accuracy:", test_acc)
print("Test macro F1:", test_macro_f1)
print("Test weighted F1:", test_weighted_f1)
print("Test balanced accuracy:", test_balanced_acc)

print("Classification report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=target_names,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

Best validation macro F1: 0.3968317050284263
Best parameters: {'model_type': 'RandomForest', 'selection_metric': 'macro_f1', 'n_estimators': 200, 'max_depth': 5, 'min_samples_leaf': 10, 'max_features': 'log2', 'class_weight': 'balanced', 'val_accuracy': 0.40476190476190477, 'val_macro_f1': 0.3968317050284263, 'val_balanced_accuracy': 0.42328042328042326}
Test accuracy: 0.38492063492063494
Test macro F1: 0.3338521893129691
Test weighted F1: 0.4275536739552725
Test balanced accuracy: 0.3412698412698412
Classification report:
              precision    recall  f1-score   support

 mild_stress       0.48      0.33      0.39       105
      stress       0.59      0.45      0.51       126
 high_stress       0.06      0.24      0.10        21

    accuracy                           0.38       252
   macro avg       0.38      0.34      0.33       252
weighted avg       0.50      0.38      0.43       252

Confusion matrix:
[[35 38 32]
 [24 57 45]
 [14  2  5]]


# Multilayer Perceptron

In [18]:
K.clear_session()
y_v = label
y_v = to_categorical(y_v)
x_train, x_test, y_train, y_test = train_test_split(
    data, y_v, test_size=0.2, random_state=1)
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.25, random_state=1)

In [19]:
def model_builder(hp):
    model = models.Sequential()
    model.add(Input(shape=(x_train.shape[1],)))

    for i in range(hp.Int('layers', 2, 6)):
        model.add(Dense(units=hp.Int('units_' + str(i), 32, 1024, step=32),
                        activation=hp.Choice('act_' + str(i), ['relu', 'sigmoid'])))

    model.add(Dense(v.N_CLASSES, activation='softmax', name='out'))

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(optimizer=opt.adam_v2.Adam(learning_rate=hp_learning_rate),
                  loss="binary_crossentropy",
                  metrics=['accuracy'])
    return model

In [20]:
tuner = RandomSearch(
    model_builder,
    objective='val_accuracy',
    max_trials=15,
    executions_per_trial=2,
    overwrite=True
)

AttributeError: module 'keras.optimizers' has no attribute 'adam_v2'

In [ ]:
tuner.search(x_train, y_train, epochs=50, validation_data=[x_val, y_val])

Trial 15 Complete [00h 01m 08s]
val_accuracy: 0.5450000166893005

Best val_accuracy So Far: 0.5541666746139526
Total elapsed time: 00h 14m 40s
INFO:tensorflow:Oracle triggered exit


In [ ]:
model = tuner.get_best_models(num_models=1)[0]

In [ ]:
y_pred = model.predict(x_test)
y_true = y_test
y_pred = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_true, axis=1)

2022-12-11 13:24:08.707790: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


In [ ]:
print(metrics.classification_report(y_true, y_pred))
print(metrics.confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.52      0.90      0.66       311
           1       0.52      0.11      0.19       289

    accuracy                           0.52       600
   macro avg       0.52      0.51      0.43       600
weighted avg       0.52      0.52      0.43       600

[[281  30]
 [256  33]]
